# EDA - Home Credit Default Risk

Análise exploratória simples com **ydata-profiling** sobre as colunas usadas pelo pipeline (`RAX_COLUMNS`).

- Conjunto análisado: `application_train.csv` (dados brutos) e comparação Referência vs Produção
- Por minimização (LGPD), apenas as colunas necessárias ao modelo são carregadas
- Os relatórios HTML são salvos em `reports/`e não são versionados

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd
from ydata_profiling import ProfileReport

from src.data.home_credit import load_application_train
from src.features.build import RAW_COLUMNS

REPORTS_DIR = ROOT / "reports"
REPORTS_DIR.mkdir(exist_ok=True)
SEED = 42

In [ ]:
raw = load_application_train()
print(f"Linhas: {len(raw):,} | Colunas: {len(raw.columns[1:]):,}")
raw.head()

In [ ]:
summary = pd.DataFrame(
    {
        "dtype": raw.dtypes.astype(str),
        "nulos_%": raw.isnull().mean().mul(100).round(2),
        "unicos": raw.nunique(),
    }
).sort_values("nulos_%", ascending=False)

summary

In [ ]:
# balancear o targe
raw["TARGET"].value_counts(normalize=True).mul(100).round(2).rename("percent")

## Profiling dos dados brutos

Amostra reproduzível de 50 mil registros para manter o relatório leve. O ID é removido, pois não carrega informação analítica.

In [ ]:
sample = raw.drop(columns=[ID_COLUMN]).sample(n=50_000, random_state=SEED)

profile = ProfileReport(
    sample,
    title="Home Credit Default Risk - EDA",
    minimal=False,
    explorative=True,
    correlations={"auto": {"calculate": True}},
    interactions={"continuous": False},
)
profile.to_file(REPORTS_DIR / "01_eda_home_credit.html")

## Comparação Referência × Produção

Confirma que, antes da simulação de drift, as duas bases têm distribuições parecidas. Requer `uv run python -m src.pipelines.prepare_data` executado antes.

In [ ]:
processed = ROOT / "data" / "processed"
reference= pd.read_parquet(processed / "reference.parquet").drop(columns=[ID_COLUMN])
production = pd.read_parquet(processed / "production_base.parquet").drop(columns=[ID_COLUMN])

ref_profile = ProfileReport(
    reference.sample(n=20_000, random_state=SEED),
    title="Reference",
    minimal=True)

prod_profile = ProfileReport(
    production.sample(n=20_000, random_state=SEED),
    title="Production",
    minimal=True
)

comparison = ref_profile.compare(prod_profile)
comparison.to_file(REPORTS_DIR / "01_eda_home_credit_comparison.html")